# Revue en aveugle de la classe « Tomato leaf »
 
**Objectif :** Vérifier visuellement, sans aucune étiquette ni nom de fichier visible, les images issues de la classe générique `Tomato leaf` de PlantDoc afin de les reclasser parmi les 5 classes cibles ou de les exclure avant le gel du test externe (E5).

In [3]:
import csv
import random
from pathlib import Path
%pip install -q matplotlib

import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, clear_output

# --- CONFIGURATION ---
# Adapter le chemin vers le dossier contenant les images "Tomato leaf" de PlantDoc
IMAGES_DIR = Path("data/PlantDoc/train/Tomato leaf")
OUTPUT_DIR = Path("audit/pd")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASSES = {
    "0": "alternariose_precoce",
    "1": "cladosporiose",
    "2": "mildiou_tardif",
    "3": "saine",
    "4": "septoriose",
    "x": "exclure_ambigu",
}


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## 1. Fonctions de chargement, d'annotation en aveugle et de consolidation

In [4]:
def load_images(folder: Path) -> list[Path]:
    exts = {".jpg", ".jpeg", ".png", ".bmp"}
    files = [p for p in folder.rglob("*") if p.suffix.lower() in exts]
    if not files:
        raise FileNotFoundError(f"Aucune image trouvée dans {folder.resolve()}")
    return sorted(files)


def save_individual_csv(path: Path, records: dict):
    if not records:
        return
    fieldnames = list(next(iter(records.values())).keys())
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in sorted(records.values(), key=lambda r: r["image_id"]):
            writer.writerow(row)


def run_blind_annotation_notebook(annotator: str, images: list[Path]) -> Path:
    annotator = annotator.strip().lower()
    if annotator not in {"aymane", "adnane"}:
        raise ValueError("L'annotateur doit être 'aymane' ou 'adnane'.")

    out_csv = OUTPUT_DIR / f"blind_review_{annotator}.csv"
    done = {}

    # Reprise automatique si une session a été interrompue
    if out_csv.exists():
        with out_csv.open("r", encoding="utf-8", newline="") as f:
            for row in csv.DictReader(f):
                done[row["image_id"]] = row

    # Ordre aléatoire propre à chaque annotateur pour éviter tout biais de séquence
    rng = random.Random(42 if annotator == "aymane" else 99)
    shuffled = images.copy()
    rng.shuffle(shuffled)

    legend = " | ".join([f"[{k}] {v}" for k, v in CLASSES.items()]) + " | [q] Sauvegarder et quitter"

    for idx, img_path in enumerate(shuffled, 1):
        img_id = img_path.name
        if img_id in done:
            continue

        clear_output(wait=True)
        print(f"Annotateur : {annotator.upper()} | Progression : {len(done)}/{len(shuffled)}")
        print(legend)
        print("-" * 80)

        img = Image.open(img_path).convert("RGB")
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(img)
        ax.axis("off")
        # Aucun nom de fichier ni étiquette affiché à l'écran
        ax.set_title(f"Image {idx}/{len(shuffled)}")
        plt.show()

        while True:
            choice = input(f"[{idx}/{len(shuffled)}] Classe (0-4, x=exclure, q=quitter) : ").strip().lower()
            if choice == "q":
                clear_output(wait=True)
                save_individual_csv(out_csv, done)
                print(f"Session sauvegardée dans {out_csv} ({len(done)}/{len(shuffled)} images).")
                return out_csv
            if choice in CLASSES:
                comment = input("Note optionnelle (Entrée pour passer) : ").strip()
                done[img_id] = {
                    "image_id": img_id,
                    "relative_path": img_path.as_posix(),
                    f"label_{annotator}": CLASSES[choice],
                    f"note_{annotator}": comment,
                }
                save_individual_csv(out_csv, done)
                break
            print("Choix invalide. Utilisez 0, 1, 2, 3, 4, x ou q.")

    clear_output(wait=True)
    save_individual_csv(out_csv, done)
    print(f"Annotation terminée pour {annotator} -> {out_csv}")
    return out_csv

## 2. Session d'annotation en aveugle
Exécuter la cellule ci-dessous une première fois avec `ANNOTATOR = "aymane"`, puis une seconde fois avec `ANNOTATOR = "adnane"`.

In [ ]:
ANNOTATOR = "aymane"  # Changer en "adnane" pour la seconde revue indépendante

# Recherche du dossier réel, même si le notebook utilise un mauvais dossier courant
candidats = [IMAGES_DIR]

for racine in [Path.cwd(), Path.cwd().parent]:
    candidats.extend(
        p for p in racine.rglob("*")
        if p.is_dir() and p.name.casefold() == "tomato leaf"
    )

imgs = []
dossier_trouve = None

for dossier in candidats:
    if dossier.exists():
        fichiers = [
            p for p in dossier.rglob("*")
            if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}
        ]
        if fichiers:
            dossier_trouve = dossier
            imgs = sorted(fichiers)
            break

if not imgs:
    raise FileNotFoundError(
        "Aucune image trouvée. Vérifiez que le dossier PlantDoc contient "
        "un dossier nommé 'Tomato leaf' avec des fichiers image."
    )

print(f"Dossier utilisé : {dossier_trouve.resolve()}")
print(f"Images trouvées : {len(imgs)}")

run_blind_annotation_notebook(ANNOTATOR, imgs)
run_blind_annotation_notebook(ANNOTATOR, imgs)

Annotation terminée pour adnane -> audit\pd\blind_review_adnane.csv


WindowsPath('audit/pd/blind_review_adnane.csv')

## 3. Consolidation et génération du journal des désaccords
Une fois les deux fichiers individuels (`blind_review_aymane.csv` et `blind_review_adnane.csv`) générés, exécuter cette cellule pour produire le fichier de preuve final `audit/pd/blind_review_tomato_leaf.csv`.

In [9]:
def merge_and_log_disagreements():
    f_aymane = OUTPUT_DIR / "blind_review_aymane.csv"
    f_adnane = OUTPUT_DIR / "blind_review_adnane.csv"
    if not (f_aymane.exists() and f_adnane.exists()):
        print("En attente du second annotateur pour générer le journal des désaccords.")
        return None

    with f_aymane.open("r", encoding="utf-8") as f:
        aymane_data = {r["image_id"]: r for r in csv.DictReader(f)}
    with f_adnane.open("r", encoding="utf-8") as f:
        adnane_data = {r["image_id"]: r for r in csv.DictReader(f)}

    common_ids = sorted(set(aymane_data) & set(adnane_data))
    merged_path = OUTPUT_DIR / "blind_review_tomato_leaf.csv"
    disagreements = 0

    with merged_path.open("w", encoding="utf-8", newline="") as f:
        fieldnames = [
            "image_id",
            "relative_path",
            "label_origine",
            "label_aymane",
            "label_adnane",
            "accord",
            "decision_finale",
            "justification_desaccord",
        ]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for img_id in common_ids:
            l_ay = aymane_data[img_id]["label_aymane"]
            l_ad = adnane_data[img_id]["label_adnane"]
            accord = l_ay == l_ad
            if not accord:
                disagreements += 1

            writer.writerow({
                "image_id": img_id,
                "relative_path": aymane_data[img_id]["relative_path"],
                "label_origine": "Tomato leaf",
                "label_aymane": l_ay,
                "label_adnane": l_ad,
                "accord": "OUI" if accord else "NON",
                "decision_finale": l_ay if accord else "A_ARBITRER",
                "justification_desaccord": "",
            })

    taux_accord = ((len(common_ids) - disagreements) / len(common_ids) * 100) if common_ids else 0
    print("=== BILAN DE CONSOLIDATION (R3) ===")
    print(f"Images comparées : {len(common_ids)}")
    print(f"Accords directs  : {len(common_ids) - disagreements} ({taux_accord:.1f} %)")
    print(f"Désaccords       : {disagreements} (marqués 'A_ARBITRER')")
    print(f"Fichier généré   : {merged_path}")
    return merged_path

merge_and_log_disagreements()

=== BILAN DE CONSOLIDATION (R3) ===
Images comparées : 8
Accords directs  : 8 (100.0 %)
Désaccords       : 0 (marqués 'A_ARBITRER')
Fichier généré   : audit\pd\blind_review_tomato_leaf.csv


WindowsPath('audit/pd/blind_review_tomato_leaf.csv')